# Валидация годовых выгрузок и аналитика клиентов (2023–2025)

Отдельная тетрадка (без изменений `crm_inn_crosscheck.ipynb`).

Что делает:
1. Проверяет валидность 3 годовых выгрузок клиентов (`Выгрузка 2023/2024/2025.csv`).
2. Сравнивает объединение их ИНН с `Выгрузка_CRM_все_клиенты.csv`.
3. Проверяет утверждение: `fp_by_inn = crm_last_version \\ crm_all_clients`.
4. Считает по годам и сегментам:
   - всех уникальных клиентов,
   - уникальных проблемных клиентов (ФП/СФП по событию в соответствующем году),
   - долю проблемных среди всех.

In [ ]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_rows", 300)
pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 220)

# ====== Базовые пути ======
DATA_DIR = Path("/home/jovyan/documents/DMUKP_FP_DEF/data")
YEAR_DIR = DATA_DIR / "uniq_clients"

YEAR_FILES = {
    2023: YEAR_DIR / "Выгрузка 2023.csv",
    2024: YEAR_DIR / "Выгрузка 2024.csv",
    2025: YEAR_DIR / "Выгрузка 2025.csv",
}

CRM_ALL_CLIENTS_PATH = DATA_DIR / "Выгрузка_CRM_все_клиенты.csv"
CRM_LAST_VERSION_PATH = DATA_DIR / "crm_last_version.csv"
FP_BY_INN_PATH = DATA_DIR / "Выгрузка_ФП_по_ИНН.xlsx"

DATE_FROM = pd.Timestamp("2023-01-01")
DATE_TO = pd.Timestamp("2025-12-31")
ALLOWED_SOURCES = ["Н2.0", "Справочник CRM-системы", "CRM-система"]

SEGMENT_MAP = {
    "ДМСБ (микро)": "МкБ",
    "ДМСБ (малый)": "МСБ",
    "ДМСБ (средний)": "МСБ",
    "ДКБ": "ККБ",
}
SEG_ORDER = ["МкБ", "МСБ", "ККБ"]


def normalize_inn(s):
    """Тот же normalize_inn, что в crm_inn_crosscheck.ipynb"""
    if pd.isna(s):
        return None
    s = str(s).strip()
    if s.endswith(".0"):
        s = s[:-2]
    s = s.lstrip("0") or "0"
    if len(s) <= 10:
        return s.zfill(10)
    return s.zfill(12)


def pick_col(df: pd.DataFrame, aliases, label: str) -> str:
    low_map = {c.lower().strip(): c for c in df.columns}
    for a in aliases:
        if a.lower() in low_map:
            return low_map[a.lower()]
    raise KeyError(f"В {label} не найдена колонка из списка: {aliases}")


def read_semicolon_keep_all(path: Path, skip_rows: int = 33, sep: str = ";") -> pd.DataFrame:
    """Читает CSV после skip_rows без потери строк (как в crm_inn_crosscheck)."""
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        lines = f.readlines()

    if len(lines) <= skip_rows:
        raise ValueError(f"Недостаточно строк после skip_rows={skip_rows}: {path}")

    header = lines[skip_rows].rstrip("\n")
    columns = header.split(sep)
    ncols = len(columns)

    comment_candidates = ["comment", "COMMENT", "COMMENT_TEXT"]
    comment_idx = next((columns.index(c) for c in comment_candidates if c in columns), None)

    records = []
    for raw_line in lines[skip_rows + 1 :]:
        raw = raw_line.rstrip("\n")
        parts = raw.split(sep)

        if len(parts) == ncols:
            records.append(parts)
            continue

        if len(parts) > ncols and comment_idx is not None:
            extra = len(parts) - ncols
            merged_comment = sep.join(parts[comment_idx : comment_idx + extra + 1])
            repaired = parts[:comment_idx] + [merged_comment] + parts[comment_idx + extra + 1 :]
            if len(repaired) == ncols:
                records.append(repaired)
                continue

        if len(parts) < ncols:
            records.append(parts + [""] * (ncols - len(parts)))
            continue

        tail = sep.join(parts[ncols - 1 :])
        repaired = parts[: ncols - 1] + [tail]
        records.append(repaired)

    return pd.DataFrame(records, columns=columns)


def load_csv_robust(path: Path, prefer_skip33: bool = False) -> pd.DataFrame:
    """Устойчивое чтение CSV: несколько стратегий, берем вариант с большим числом колонок."""
    candidates = []

    if prefer_skip33:
        try:
            candidates.append(read_semicolon_keep_all(path=path, skip_rows=33, sep=";"))
        except Exception:
            pass

    for kwargs in [
        {"dtype": str, "low_memory": False},
        {"dtype": str, "sep": ";", "engine": "python"},
        {"dtype": str, "sep": ",", "engine": "python"},
        {"dtype": str, "sep": None, "engine": "python"},
    ]:
        try:
            candidates.append(pd.read_csv(path, **kwargs))
        except Exception:
            pass

    if not candidates:
        raise RuntimeError(f"Не удалось прочитать CSV: {path}")

    best = max(candidates, key=lambda df: df.shape[1])
    best.columns = [str(c).strip() for c in best.columns]
    return best


def normalize_segment_col(series: pd.Series) -> pd.Series:
    s = series.astype(str).str.strip()
    mapped = s.map(SEGMENT_MAP)
    mapped = mapped.where(mapped.notna(), s.where(s.isin(SEG_ORDER)))
    return mapped


In [ ]:
print("Проверка наличия файлов:")
for y, p in YEAR_FILES.items():
    print(f"year {y}: {p} -> {'OK' if p.exists() else 'MISSING'}")
print(f"crm_all_clients: {CRM_ALL_CLIENTS_PATH} -> {'OK' if CRM_ALL_CLIENTS_PATH.exists() else 'MISSING'}")
print(f"crm_last_version: {CRM_LAST_VERSION_PATH} -> {'OK' if CRM_LAST_VERSION_PATH.exists() else 'MISSING'}")
print(f"fp_by_inn: {FP_BY_INN_PATH} -> {'OK' if FP_BY_INN_PATH.exists() else 'MISSING'}")

# ====== 1) Годовые выгрузки ======
year_frames = {}
year_stats = []

for year, path in YEAR_FILES.items():
    raw = load_csv_robust(path, prefer_skip33=False)

    inn_col = pick_col(raw, ["X_INN", "inn", "ИНН"], f"year_{year}")
    seg_col = pick_col(raw, ["X_AREA_RESP", "segment", "Сегмент"], f"year_{year}")

    df = raw.copy()
    df["inn"] = df[inn_col].apply(normalize_inn)
    df["segment"] = normalize_segment_col(df[seg_col])
    df["year"] = year

    rows_raw = len(df)
    df = df.dropna(subset=["inn", "segment"]).copy()
    rows_clean = len(df)

    year_frames[year] = df
    year_stats.append(
        {
            "year": year,
            "rows_raw": rows_raw,
            "rows_after_clean": rows_clean,
            "unique_inn": int(df["inn"].nunique()),
            "unique_year_segment_inn": int(df[["year", "segment", "inn"]].drop_duplicates().shape[0]),
        }
    )

year_stats_df = pd.DataFrame(year_stats).sort_values("year").reset_index(drop=True)

print("\nСтатистика по 3 годовым выгрузкам:")
display(year_stats_df)

all_year_df = pd.concat([year_frames[y] for y in sorted(year_frames)], ignore_index=True)

# ====== 2) Контрольная база crm_all_clients ======
crm_all_raw = load_csv_robust(CRM_ALL_CLIENTS_PATH, prefer_skip33=True)
crm_all_inn_col = pick_col(crm_all_raw, ["X_INN", "inn", "ИНН"], "crm_all_clients")
crm_all_seg_col = pick_col(crm_all_raw, ["X_AREA_RESP", "segment", "Сегмент"], "crm_all_clients")

crm_all = crm_all_raw.copy()
crm_all["inn"] = crm_all[crm_all_inn_col].apply(normalize_inn)
crm_all["segment"] = normalize_segment_col(crm_all[crm_all_seg_col])
crm_all = crm_all.dropna(subset=["inn", "segment"]).copy()
crm_all = crm_all.drop_duplicates(subset=["inn"]).copy()

set_year_union = set(all_year_df["inn"].dropna().unique())
set_all_clients = set(crm_all["inn"].dropna().unique())

only_in_year_files = set_year_union - set_all_clients
only_in_all_clients = set_all_clients - set_year_union

validation_year_vs_all = pd.DataFrame([
    {"metric": "|year_union|", "value": len(set_year_union)},
    {"metric": "|crm_all_clients|", "value": len(set_all_clients)},
    {"metric": "|intersection|", "value": len(set_year_union & set_all_clients)},
    {"metric": "|only_in_year_files|", "value": len(only_in_year_files)},
    {"metric": "|only_in_all_clients|", "value": len(only_in_all_clients)},
])

print("\nВалидация годовых выгрузок против crm_all_clients:")
display(validation_year_vs_all)

print("Примеры only_in_year_files (до 30):")
display(pd.DataFrame({"inn": sorted(list(only_in_year_files))[:30]}))

print("Примеры only_in_all_clients (до 30):")
display(pd.DataFrame({"inn": sorted(list(only_in_all_clients))[:30]}))


In [ ]:
# ====== 3) Проверка fp_by_inn = crm_last_version \ crm_all_clients ======
crm_last_raw = load_csv_robust(CRM_LAST_VERSION_PATH, prefer_skip33=False)

crm_last_inn_col = pick_col(crm_last_raw, ["X_INN", "inn", "ИНН"], "crm_last_version")
crm_last_seg_col = pick_col(crm_last_raw, ["X_AREA_RESP", "segment", "Сегмент"], "crm_last_version")
crm_last_dt_col = pick_col(crm_last_raw, ["IDENTIFICATION_DATE"], "crm_last_version")
crm_last_val_col = pick_col(crm_last_raw, ["VAL"], "crm_last_version")
crm_last_type_col = pick_col(crm_last_raw, ["TYPE_FP"], "crm_last_version")
crm_last_num_col = pick_col(crm_last_raw, ["NUMBER_FP_SFP"], "crm_last_version")

crm_last = crm_last_raw.copy()
crm_last["inn"] = crm_last[crm_last_inn_col].apply(normalize_inn)
crm_last["segment"] = normalize_segment_col(crm_last[crm_last_seg_col])
crm_last["dt"] = pd.to_datetime(crm_last[crm_last_dt_col], dayfirst=True, errors="coerce")
crm_last["VAL_norm"] = crm_last[crm_last_val_col].astype(str).str.strip()
crm_last["TYPE_FP_norm"] = (
    crm_last[crm_last_type_col]
    .astype(str)
    .str.strip()
    .replace({"FP": "ФП", "SFP": "СФП"})
)
crm_last["fp_num"] = crm_last[crm_last_num_col].astype(str).str.strip()

crm_last = crm_last[
    (crm_last["dt"] >= DATE_FROM)
    & (crm_last["dt"] <= DATE_TO)
    & (crm_last["VAL_norm"].isin(ALLOWED_SOURCES))
    & (crm_last["segment"].notna())
    & (crm_last["inn"].notna())
    & (crm_last["fp_num"].notna())
    & (crm_last["TYPE_FP_norm"].isin(["ФП", "СФП"]))
].copy()

crm_last = crm_last.drop_duplicates(subset=["inn", "fp_num", "TYPE_FP_norm", crm_last_dt_col]).copy()

fp_by_inn_raw = pd.read_excel(FP_BY_INN_PATH, dtype=str)
fp_by_inn_raw.columns = [str(c).strip() for c in fp_by_inn_raw.columns]
fp_inn_col = pick_col(fp_by_inn_raw, ["X_INN", "inn", "ИНН"], "fp_by_inn")
fp_by_inn_raw["inn"] = fp_by_inn_raw[fp_inn_col].apply(normalize_inn)

set_last = set(crm_last["inn"].dropna().unique())
set_fp = set(fp_by_inn_raw["inn"].dropna().unique())
expected_fp_set = set_last - set_all_clients

only_in_fp_by_inn = set_fp - expected_fp_set
missing_from_fp_by_inn = expected_fp_set - set_fp

fp_delta_check = pd.DataFrame([
    {"metric": "|set_last|", "value": len(set_last)},
    {"metric": "|set_all_clients|", "value": len(set_all_clients)},
    {"metric": "|expected_fp_set = set_last \\ set_all_clients|", "value": len(expected_fp_set)},
    {"metric": "|fp_by_inn|", "value": len(set_fp)},
    {"metric": "fp_by_inn == expected_fp_set", "value": bool(set_fp == expected_fp_set)},
    {"metric": "|only_in_fp_by_inn|", "value": len(only_in_fp_by_inn)},
    {"metric": "|missing_from_fp_by_inn|", "value": len(missing_from_fp_by_inn)},
])

print("Проверка тождества fp_by_inn и (crm_last_version \\ crm_all_clients):")
display(fp_delta_check)

print("Примеры only_in_fp_by_inn (до 30):")
display(pd.DataFrame({"inn": sorted(list(only_in_fp_by_inn))[:30]}))

print("Примеры missing_from_fp_by_inn (до 30):")
display(pd.DataFrame({"inn": sorted(list(missing_from_fp_by_inn))[:30]}))


In [ ]:
# ====== 4) Годовые и сегментные метрики: все клиенты vs проблемные ======
crm_last["event_year"] = crm_last["dt"].dt.year.astype(int)

problem_set_by_year = {
    y: set(crm_last.loc[crm_last["event_year"] == y, "inn"].dropna().unique())
    for y in [2023, 2024, 2025]
}

all_rows = []
seg_rows = []

for y in [2023, 2024, 2025]:
    df_y = year_frames[y].copy()

    all_inn_y = set(df_y["inn"].dropna().unique())
    problem_inn_y = all_inn_y & problem_set_by_year.get(y, set())

    all_cnt = len(all_inn_y)
    prob_cnt = len(problem_inn_y)
    share = (prob_cnt / all_cnt) if all_cnt else 0.0

    all_rows.append(
        {
            "year": y,
            "all_unique_clients": all_cnt,
            "fp_sfp_unique_clients": prob_cnt,
            "share_fp_sfp_among_all": share,
        }
    )

    seg_base = df_y[["segment", "inn"]].drop_duplicates()
    for seg, seg_df in seg_base.groupby("segment"):
        seg_inn = set(seg_df["inn"].dropna().unique())
        seg_all = len(seg_inn)
        seg_prob = len(seg_inn & problem_set_by_year.get(y, set()))
        seg_share = (seg_prob / seg_all) if seg_all else 0.0
        seg_rows.append(
            {
                "year": y,
                "segment": seg,
                "all_unique_clients": seg_all,
                "fp_sfp_unique_clients": seg_prob,
                "share_fp_sfp_among_all": seg_share,
            }
        )

yearly_summary = pd.DataFrame(all_rows).sort_values("year").reset_index(drop=True)
yearly_segment_summary = (
    pd.DataFrame(seg_rows)
    .sort_values(["year", "segment"], key=lambda s: s.map({"МкБ": 0, "МСБ": 1, "ККБ": 2}).fillna(99) if s.name == "segment" else s)
    .reset_index(drop=True)
)

print("Итог по годам (все vs проблемные клиенты):")
display(yearly_summary)

print("\nИтог по годам и сегментам (все vs проблемные клиенты):")
display(yearly_segment_summary)
